# **Dados e Aprendizagem Automática** 

## **Stacking -  Tratamento 2 com Grid**
Nesta abordagem, para além do tratamento de dados onde aplicámos Label Encoding às variáveis categóricas, melhoramos a fase de Modelação com o Grid Search em todos os modelos a utilizar no stacking, Decision Trees, Random Forest e Logistic Regression.

**Imports necessários para este teste**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

%matplotlib inline

### **Preparation**
**Load the CSVs**

In [2]:
df_train = pd.read_csv('../../Datasets/training_data.csv', encoding='latin-1')
df_test = pd.read_csv('../../Datasets/test_data.csv', encoding='latin-1')

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

Train shape: (6812, 14)
Test shape: (1500, 13)


**Feature Engineering (Extração da data)**

In [3]:
def extract_date_features(df):
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek # Monday=0, Sunday=6
    df['month'] = df['record_date'].dt.month
    
    # Create "Weekend" feature
    df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
    
    # Create "Rush Hour" feature (7 da manhã até às 9 da manhã e 4 da tarde ate às 7 da tarde, podemos brincar com estas horas)
    df['is_rush_hour'] = df['hour'].apply(lambda x: 1 if (7 <= x <= 9) or (16 <= x <= 19) else 0)
    
    return df.drop(columns=['record_date'])

df_train = extract_date_features(df_train)
df_test = extract_date_features(df_test)

**Missing Values e Valores Incorretos**

In [4]:
def clean_categorical_text(df):

    # Primeiro "limpamos" a coluna 'AVERAGE CLOUDINESS'
    correcoes_erros = {
        'cï¿½u': 'ceu',      # erro especifico
        'c\u00e9u': 'ceu', # é
        'c\u00fa': 'ceu',  # ú
        'c\u00fau': 'ceu', 
        'céu': 'ceu'
    }
    # regex=True permite substituir apenas parte da frase (ex: "cï¿½u claro" -> "ceu claro")
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(correcoes_erros, regex=True)

    cloud_map = {
        'ceu claro': 'ceu_claro',
        'ceu limpo': 'ceu_claro',

        'ceu pouco nublado': 'pouco_nublado',
        'nuvens dispersas': 'pouco_nublado',
        'algumas nuvens': 'pouco_nublado',

        'nuvens quebrados': 'nublado', 
        'nuvens quebradas': 'nublado',
        'tempo nublado': 'nublado',
        'nublado': 'nublado',
    }
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].astype(str).replace(cloud_map, regex=True)
    # Tratamos também dos seus missing values
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].replace('nan', 'unknown_cloudiness')
    
    # Depois "limpamos" também a coluna da "AVERAGE RAIN"
    rain_map = {
        'chuva fraca': 'chuva_fraca',
        'chuva leve': 'chuva_fraca',
        'aguaceiros fracos': 'chuva_fraca',
        'chuvisco fraco': 'chuva_fraca',
        'chuvisco e chuva fraca': 'chuva_fraca',
        'trovoada com chuva leve': 'chuva_fraca', 

        'chuva moderada': 'chuva_moderada',
        'chuva': 'chuva_moderada',
        'aguaceiros': 'chuva_moderada',
        'trovoada com chuva': 'chuva_moderada',

        'chuva forte': 'chuva_forte',
        'chuva de intensidade pesada': 'chuva_forte',
        'chuva de intensidade pesado': 'chuva_forte'
    }
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].replace(rain_map)
    # Tratamos também dos seus missing values
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].fillna('no_rain')
    
    #df["LUMINOSITY"] = df_train["LUMINOSITY"].replace("LOW_LIGHT", "LIGHT")
    
    return df

df_train = clean_categorical_text(df_train)
df_test = clean_categorical_text(df_test)

In [5]:
df_train["AVERAGE_SPEED_DIFF"] = df_train["AVERAGE_SPEED_DIFF"].fillna("None")

**Verificação dos valores dessas colunas agora**

In [6]:
df_test['AVERAGE_CLOUDINESS'].value_counts()

AVERAGE_CLOUDINESS
unknown_cloudiness    599
ceu_claro             372
pouco_nublado         301
nublado               228
Name: count, dtype: int64

In [7]:
df_test['AVERAGE_RAIN'].value_counts()

AVERAGE_RAIN
no_rain           1360
chuva_fraca         93
chuva_moderada      47
Name: count, dtype: int64

**Drop de Colunas Redundates** 

Considera-mo-las redundantes devido a 'CITY_NAME' conter um valor constante ("Porto") e 'AVERAGE_PRECIPITATION' consistir quase apenas em zeros.

In [8]:
cols_to_drop = ['city_name', 'AVERAGE_PRECIPITATION']
df_train = df_train.drop(columns=cols_to_drop)
df_test = df_test.drop(columns=cols_to_drop)

**Handling categoric data**

In [9]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OrdinalEncoder

In [10]:
"""
AVERAGE_RAIN
"""

le_rain = LabelEncoder()
df_train["AVERAGE_RAIN"] = le_rain.fit_transform(df_train["AVERAGE_RAIN"])
df_test["AVERAGE_RAIN"] = le_rain.transform(df_test["AVERAGE_RAIN"])

In [11]:
"""
AVERAGE_CLOUDINESS
"""
le_cloud = LabelEncoder()
df_train["AVERAGE_CLOUDINESS"] = le_cloud.fit_transform(df_train["AVERAGE_CLOUDINESS"])
df_test["AVERAGE_CLOUDINESS"] = le_cloud.transform(df_test["AVERAGE_CLOUDINESS"])

In [12]:
"""
LUMINOSITY
"""
le_lu = LabelEncoder()
df_train["LUMINOSITY"] = le_lu.fit_transform(df_train["LUMINOSITY"])
df_test["LUMINOSITY"] = le_lu.transform(df_test["LUMINOSITY"])

In [13]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6812 entries, 0 to 6811
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_SPEED_DIFF       6812 non-null   object 
 1   AVERAGE_FREE_FLOW_SPEED  6812 non-null   float64
 2   AVERAGE_TIME_DIFF        6812 non-null   float64
 3   AVERAGE_FREE_FLOW_TIME   6812 non-null   float64
 4   LUMINOSITY               6812 non-null   int64  
 5   AVERAGE_TEMPERATURE      6812 non-null   float64
 6   AVERAGE_ATMOSP_PRESSURE  6812 non-null   float64
 7   AVERAGE_HUMIDITY         6812 non-null   float64
 8   AVERAGE_WIND_SPEED       6812 non-null   float64
 9   AVERAGE_CLOUDINESS       6812 non-null   int64  
 10  AVERAGE_RAIN             6812 non-null   int64  
 11  hour                     6812 non-null   int32  
 12  day_of_week              6812 non-null   int32  
 13  month                    6812 non-null   int32  
 14  is_weekend              

In [14]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   AVERAGE_FREE_FLOW_SPEED  1500 non-null   float64
 1   AVERAGE_TIME_DIFF        1500 non-null   float64
 2   AVERAGE_FREE_FLOW_TIME   1500 non-null   float64
 3   LUMINOSITY               1500 non-null   int64  
 4   AVERAGE_TEMPERATURE      1500 non-null   float64
 5   AVERAGE_ATMOSP_PRESSURE  1500 non-null   float64
 6   AVERAGE_HUMIDITY         1500 non-null   float64
 7   AVERAGE_WIND_SPEED       1500 non-null   float64
 8   AVERAGE_CLOUDINESS       1500 non-null   int64  
 9   AVERAGE_RAIN             1500 non-null   int64  
 10  hour                     1500 non-null   int32  
 11  day_of_week              1500 non-null   int32  
 12  month                    1500 non-null   int32  
 13  is_weekend               1500 non-null   int64  
 14  is_rush_hour            

### **Modeling**

Select features and target

In [15]:
X_train = df_train.drop(columns=["AVERAGE_SPEED_DIFF"])
y_train = df_train["AVERAGE_SPEED_DIFF"]

In [16]:
X_test = df_test

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

**Grid Search**

Hyperparameter Tuning with Grid Search - Check what is the best estimator and parameters

Random Forest

In [17]:
rf = RandomForestClassifier(random_state=2022)

param_grid_rf = {
    'n_estimators': [100, 200],         # Number of trees in the forest
    'max_depth': [10, 20, None],        # Maximum depth of the tree
    'min_samples_split': [2, 5],        # Minimum samples required to split an internal node
    'criterion': ['gini', 'entropy']    # Gini or Entropy
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=3,          # 3-Fold Cross Validation (speeds it up compared to 5)
    verbose=0,
    n_jobs=-1      # Use all processor cores
)

grid_rf.fit(X_train, y_train)

print("Best Parameters:", grid_rf.best_params_)
print("Best CV Accuracy:", grid_rf.best_score_)

best_rf = grid_rf.best_estimator_

Best Parameters: {'criterion': 'entropy', 'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 100}
Best CV Accuracy: 0.7996169541127321


Decision Trees

In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

dt_model = DecisionTreeClassifier(random_state = 2022)
dt_model.fit(X_train, y_train)

ccp_alphas = dt_model.cost_complexity_pruning_path(X_train, y_train)["ccp_alphas"]

max_depth_tree = dt_model.get_depth()
print(max_depth_tree)
print(dt_model.get_n_leaves())

23
1108


In [19]:
ccp_alphas

array([0.00000000e+00, 9.36114444e-05, 9.37887388e-05, 9.54198473e-05,
       9.57842439e-05, 9.61495538e-05, 9.66582816e-05, 9.67668864e-05,
       9.69518698e-05, 9.69848298e-05, 1.21833819e-04, 1.22333138e-04,
       1.22333138e-04, 1.22333138e-04, 1.25828370e-04, 1.25828370e-04,
       1.25828370e-04, 1.28449794e-04, 1.28449794e-04, 1.28449794e-04,
       1.28449794e-04, 1.28449794e-04, 1.30331843e-04, 1.30488680e-04,
       1.31347158e-04, 1.32119789e-04, 1.32818835e-04, 1.33342156e-04,
       1.33454332e-04, 1.33454332e-04, 1.33454332e-04, 1.33454332e-04,
       1.34566451e-04, 1.34566451e-04, 1.34566451e-04, 1.35507475e-04,
       1.35507475e-04, 1.35507475e-04, 1.35507475e-04, 1.35507475e-04,
       1.35566392e-04, 1.36314068e-04, 1.36314068e-04, 1.36314068e-04,
       1.36314068e-04, 1.37328813e-04, 1.37624780e-04, 1.37624780e-04,
       1.38164485e-04, 1.38164485e-04, 1.38644223e-04, 1.39073462e-04,
       1.39073462e-04, 1.39073462e-04, 1.39271572e-04, 1.39422673e-04,
      

In [20]:
# Choose a smaller subset
ccp_alphas = np.unique(ccp_alphas)
ccp_alphas = ccp_alphas[::int(len(ccp_alphas)/15)+1] # use steps and +- 20 values
ccp_alphas

array([0.        , 0.00013816, 0.0001468 , 0.00019962, 0.00022773,
       0.00024772, 0.00026691, 0.00028918, 0.00033032, 0.000367  ,
       0.0004022 , 0.00045926, 0.00053643, 0.00077997, 0.00151801])

In [21]:
# Define the params to involve (Brincar com isto)
param_grid_dt = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [1,2,4,8,16,32, None],
    'ccp_alpha': ccp_alphas
}

# Model
dt = DecisionTreeClassifier(random_state=2022)

# Grid Search
grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    cv=3,
    n_jobs=-1,
    verbose=0 
)

grid_dt.fit(X_train, y_train)

print("Best Parameters:", grid_dt.best_params_)
print("Best CV Accuracy:", grid_dt.best_score_)

# Get the best model
best_dt = grid_dt.best_estimator_


Best Parameters: {'ccp_alpha': np.float64(0.0015180063623986065), 'criterion': 'gini', 'max_depth': 8}
Best CV Accuracy: 0.7692303713230277


Logistic Regression

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train)  
X_test_scaled = scaler.transform(X_test) 

# Logistic Regression 
lr_model = LogisticRegression(
    random_state=2022,
    max_iter=2000,
    n_jobs=-1)

param_grid_lr = {
    'C': [0.01, 0.1, 1.0],
    'solver': ['lbfgs']
}
grid_lr = GridSearchCV(
    lr_model,
    param_grid_lr, 
    cv=3, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=0)
grid_lr.fit(X_train_scaled, y_train)

print("Best parameteres:", grid_lr.best_params_)
print("Best CV accuracy:", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_

Best parameteres: {'C': 1.0, 'solver': 'lbfgs'}
Best CV accuracy: 0.7726069169396935


**Stacking**

In [23]:
from sklearn.ensemble import StackingClassifier

# Best estimators from grid searches
estimators = [
    ('dt', best_dt),
    ('rf', best_rf),
    ('lr', best_lr)
]

final_estimator = LogisticRegression(
    random_state=2022,
    max_iter=2000,
    n_jobs=-1
)

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    passthrough=False, # When false, only the predictions of the estimators will be used.
                        # If true, the final_estimator is trained on the predictions as well as the original training data.
    n_jobs=-1
)

stack_param_grid = {
    'final_estimator__C': [0.01, 0.1, 1.0]
}

grid_stack = GridSearchCV(
    estimator=stack,
    param_grid=stack_param_grid,
    cv=3,
    scoring='accuracy',
    verbose=0,
    n_jobs=-1,
)

# Train
grid_stack.fit(X_train_scaled, y_train)

print("Stack best params:", grid_stack.best_params_)
print("Stack best CV accuracy:", grid_stack.best_score_)

best_stack = grid_stack.best_estimator_

Stack best params: {'final_estimator__C': 1.0}
Stack best CV accuracy: 0.8068109619404726


**Validation Block**

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Cópias
X_val_copy = X_train.copy()
y_val_copy = y_train.copy()

# criar conjunto de val
X_tr, X_val, y_tr, y_val = train_test_split(
    X_val_copy, y_val_copy, test_size=0.2, random_state=2022, stratify=y_val_copy
)

X_tr_scaled = scaler.transform(X_tr)
X_val_scaled = scaler.transform(X_val)

# Treinar modelo só para validar
clf_val = best_stack # Validar com o melhor modelo dado pelo GridSearch
clf_val.fit(X_tr_scaled, y_tr)

# Prever na validação
val_preds = clf_val.predict(X_val_scaled)

# Mostrar accuracy
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification Report:\n", classification_report(y_val, val_preds))


Validation Accuracy: 0.8195157740278797

Classification Report:
               precision    recall  f1-score   support

        High       0.78      0.78      0.78       213
         Low       0.73      0.76      0.74       284
      Medium       0.82      0.78      0.80       330
        None       0.90      0.90      0.90       440
   Very_High       0.85      0.82      0.84        96

    accuracy                           0.82      1363
   macro avg       0.81      0.81      0.81      1363
weighted avg       0.82      0.82      0.82      1363



**Predict**

Predict on test set

In [25]:
clf = best_stack
clf.fit(X_train_scaled, y_train)

,estimators,"[('dt', ...), ('rf', ...), ...]"
,final_estimator,LogisticRegre...om_state=2022)
,cv,None
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,criterion,'gini'
,splitter,'best'
,max_depth,8
,min_samples_split,2


In [26]:
preds = clf.predict(X_test_scaled)

Create csv

In [27]:
submission = pd.DataFrame({
    "RowId": range(1, len(preds)+1),
    "Speed_Diff": preds
})

submission.to_csv("../../results/Stacking/stack2grid.csv", index=False)
submission.head()

,RowId,Speed_Diff
0,1,None
1,2,Low
2,3,None
3,4,High
4,5,Low
